# Energy Consumption Analysis and Optimization
## BTI Campus Building - Universitas Pertahanan RI

### Using XGBoost Algorithm for Predictive Analysis

---

**Author:** Energy Optimization Team  
**Date:** 2024  
**Version:** 1.0

---

This notebook provides a complete analysis of energy consumption patterns at BTI Campus Building and generates optimization recommendations using machine learning (XGBoost).

### Contents:
1. Introduction & Setup
2. Data Loading & Exploration
3. Energy Consumption Analysis
4. XGBoost Modeling
5. Visualization
6. Optimization Recommendations
7. Export Results
8. Conclusion

## 1. Introduction & Setup

### Import Libraries and Load Configuration

In [ ]:
# Standard library imports
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Matplotlib settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

print('Libraries imported successfully!')
print(f'Analysis Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

In [ ]:
# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import project modules
import config
from src import data_input, energy_calculator, xgboost_model, visualization, optimization

# Create output directories
config.create_directories()

print(f'Project Root: {project_root}')
print(f'Electricity Tariff: Rp {config.ELECTRICITY_TARIFF:,}/kWh')
print(f'Working Days per Month: {config.WORKING_DAYS_PER_MONTH}')
print(f'Simulation Days: {config.SIMULATION_DAYS}')

## 2. Data Loading & Exploration

### Load Device Inventory from Excel Template

In [ ]:
# Load and process device inventory data
df_devices, summary = data_input.load_and_process_data()

print('\n' + '='*60)
print('DATA LOADING SUMMARY')
print('='*60)
print(f"Total Device Types: {summary['total_device_types']}")
print(f"Total Device Count: {summary['total_devices']}")
print(f"Categories: {', '.join(summary['categories'])}")
print(f"Total Power Capacity: {summary['total_power_kw']:.2f} kW")
print('='*60)

In [ ]:
# Display dataset information
print('\n--- Dataset Information ---\n')
print(f'Shape: {df_devices.shape[0]} rows x {df_devices.shape[1]} columns')
print(f'\nColumns: {list(df_devices.columns)}')
print(f'\nData Types:')
print(df_devices.dtypes)

In [ ]:
# Display first 10 rows of device inventory
print('\n--- Device Inventory (First 10 Rows) ---\n')
display(df_devices.head(10))

In [ ]:
# Summary statistics for numerical columns
print('\n--- Summary Statistics ---\n')
numeric_cols = ['Quantity', 'Power_Watt', 'Operating_Hours_Per_Day',
                'Total_Power_Watt', 'Daily_Energy_kWh', 'Monthly_Energy_kWh', 'Monthly_Cost_Rp']
display(df_devices[numeric_cols].describe().round(2))

In [ ]:
# Check for missing values
print('\n--- Missing Values Check ---\n')
missing = df_devices.isnull().sum()
if missing.sum() == 0:
    print('No missing values found in the dataset!')
else:
    print('Missing values by column:')
    print(missing[missing > 0])

## 3. Energy Consumption Analysis

### Calculate Total Energy Consumption

In [ ]:
# Calculate comprehensive consumption metrics
total_daily_kwh = df_devices['Daily_Energy_kWh'].sum()
total_monthly_kwh = df_devices['Monthly_Energy_kWh'].sum()
total_annual_kwh = total_monthly_kwh * 12

# Cost calculations
daily_cost = total_daily_kwh * config.ELECTRICITY_TARIFF
monthly_cost = total_monthly_kwh * config.ELECTRICITY_TARIFF
annual_cost = total_annual_kwh * config.ELECTRICITY_TARIFF

print('\n' + '='*60)
print('ENERGY CONSUMPTION SUMMARY')
print('='*60)
print(f'\nDaily Consumption:    {total_daily_kwh:,.2f} kWh')
print(f'Monthly Consumption:  {total_monthly_kwh:,.2f} kWh')
print(f'Annual Projection:    {total_annual_kwh:,.2f} kWh ({total_annual_kwh/1000:.2f} MWh)')
print(f'\nDaily Cost:           Rp {daily_cost:,.0f}')
print(f'Monthly Cost:         Rp {monthly_cost:,.0f}')
print(f'Annual Cost:          Rp {annual_cost:,.0f} (Rp {annual_cost/1_000_000:.2f} Million)')
print('='*60)

In [ ]:
# Consumption breakdown by device category
category_breakdown = energy_calculator.get_consumption_by_category(df_devices)

print('\n--- Energy Consumption by Category ---\n')
display(category_breakdown)

In [ ]:
# Top 10 energy consuming devices
top_consumers = energy_calculator.get_top_consumers(df_devices, n=10)

print('\n--- Top 10 Energy Consuming Devices ---\n')
display(top_consumers[['Device_Name', 'Device_Category', 'Quantity',
                       'Daily_Energy_kWh', 'Monthly_Energy_kWh', 'Monthly_Cost_Rp']])

## 4. XGBoost Modeling

### Generate Synthetic Training Data and Train Predictive Model

In [ ]:
# Generate synthetic data for model training
print('Generating synthetic training data...')
synthetic_data = xgboost_model.generate_synthetic_data(df_devices, num_days=config.SIMULATION_DAYS)

print(f'\n--- Synthetic Data Summary ---')
print(f'Total records: {len(synthetic_data):,}')
print(f"Time range: {synthetic_data['datetime'].min()} to {synthetic_data['datetime'].max()}")
print(f'\nSample data:')
display(synthetic_data.head(10))

In [ ]:
# Feature engineering explanation
print('\n--- Feature Engineering ---\n')
print('Features used for prediction:')
print('  - hour: Hour of day (0-23)')
print('  - day_of_week: Day of week (0=Monday, 6=Sunday)')
print('  - month: Month (1-12)')
print('  - temperature: Ambient temperature (C)')
print('  - occupancy_rate: Building occupancy (%)')
print('  - is_weekend: Weekend indicator (0/1)')
print('  - is_working_hour: Working hours indicator (0/1)')
print('  - active_devices_count: Number of active devices')
print('  - Cyclic encodings (sin/cos) for time features')
print('  - Interaction features')

In [ ]:
# Prepare data for training
print('Preparing data for training...')
X_train, X_test, y_train, y_test, feature_names = xgboost_model.prepare_data(synthetic_data)

print(f'\n--- Train/Test Split ---')
print(f'Training samples: {len(X_train):,}')
print(f'Test samples: {len(X_test):,}')
print(f'Test ratio: {config.MODEL_TEST_SIZE * 100:.0f}%')
print(f'\nFeatures ({len(feature_names)}): {feature_names}')

In [ ]:
# Train XGBoost model
print('Training XGBoost model...')
print(f'\nHyperparameters:')
for param, value in config.XGBOOST_PARAMS.items():
    print(f'  - {param}: {value}')

model = xgboost_model.train_model(X_train, y_train)
print('\nModel training complete!')

In [ ]:
# Model evaluation
print('Evaluating model performance...')
metrics = xgboost_model.evaluate_model(model, X_test, y_test)
y_pred = xgboost_model.predict(model, X_test)

print('\n' + '='*60)
print('MODEL EVALUATION METRICS')
print('='*60)
print(f"\nRoot Mean Square Error (RMSE): {metrics['rmse']:.4f} kWh")
print(f"Mean Absolute Error (MAE):     {metrics['mae']:.4f} kWh")
print(f"Mean Absolute % Error (MAPE):  {metrics['mape']:.2f}%")
print(f"R2 Score: {metrics['r2']:.4f}")
print('='*60)

if metrics['r2'] >= 0.85:
    print('\nModel achieves target R2 >= 0.85')
else:
    print(f"\nModel R2 ({metrics['r2']:.4f}) is below target (0.85)")

In [ ]:
# Feature importance
importance_df = xgboost_model.get_feature_importance(model, feature_names)

print('\n--- Feature Importance ---\n')
display(importance_df.head(10))

## 5. Visualization

### Generate Publication-Ready Figures

In [ ]:
# Create model results dictionary for visualization
model_results = {
    'model': model,
    'metrics': metrics,
    'feature_importance': importance_df,
    'predictions': y_pred,
    'actuals': y_test,
    'synthetic_data': synthetic_data,
    'feature_names': feature_names
}

In [ ]:
# Top 10 Energy Consumers
print('--- Top 10 Energy Consuming Devices ---')
fig, _ = visualization.plot_consumption_by_device(df_devices, n_devices=10, save=True, show=True)

In [ ]:
# Energy Consumption by Category (Pie Chart)
print('--- Energy Consumption by Category ---')
fig, _ = visualization.plot_consumption_pie_chart(df_devices, save=True, show=True)

In [ ]:
# Prediction vs Actual
print('--- XGBoost Prediction vs Actual ---')
fig, _ = visualization.plot_prediction_vs_actual(y_test, y_pred, save=True, show=True)

In [ ]:
# Feature Importance
print('--- XGBoost Feature Importance ---')
fig, _ = visualization.plot_feature_importance(importance_df, save=True, show=True)

In [ ]:
# Daily Consumption Pattern
print('--- Daily Energy Consumption Pattern ---')
fig, _ = visualization.plot_daily_pattern(synthetic_data, save=True, show=True)

In [ ]:
# Consumption Distribution by Category
print('--- Consumption Distribution by Category ---')
fig, _ = visualization.plot_consumption_distribution(df_devices, save=True, show=True)

In [ ]:
# Monthly Consumption by Category
print('--- Monthly Consumption by Category ---')
fig, _ = visualization.plot_monthly_consumption(df_devices, save=True, show=True)

In [ ]:
# Device Operation Schedule Heatmap
print('--- Device Operation Schedule Heatmap ---')
fig, _ = visualization.plot_heatmap_schedule(df_devices, save=True, show=True)

## 6. Optimization Recommendations

### Generate Energy Saving Recommendations

In [ ]:
# Generate recommendations
print('Generating optimization recommendations...')
recommendations = optimization.generate_recommendations(df_devices)

print(f'\n--- Optimization Recommendations ({len(recommendations)} devices) ---\n')
display(recommendations[['Device', 'Category', 'Current_Monthly_kWh',
                        'Recommendation', 'Potential_Savings_kWh',
                        'Savings_Percentage', 'ROI_Months']].head(10))

In [ ]:
# Calculate total savings
total_savings = optimization.calculate_total_savings(recommendations)

print('\n' + '='*60)
print('TOTAL POTENTIAL SAVINGS SUMMARY')
print('='*60)
print(f"\nCurrent Monthly Consumption:    {total_savings['current_monthly_consumption_kwh']:,.2f} kWh")
print(f"Current Monthly Cost:           Rp {total_savings['current_monthly_cost_rp']:,.0f}")
print(f"\nPotential Monthly Savings:      {total_savings['potential_monthly_savings_kwh']:,.2f} kWh")
print(f"Potential Monthly Cost Savings: Rp {total_savings['potential_monthly_savings_rp']:,.0f}")
print(f"Potential Annual Savings:       {total_savings['potential_annual_savings_kwh']:,.2f} kWh")
print(f"Potential Annual Cost Savings:  Rp {total_savings['potential_annual_savings_rp']:,.0f}")
print(f"\nTotal Implementation Cost:      Rp {total_savings['total_implementation_cost_rp']:,.0f}")
print(f"Overall Savings Percentage:     {total_savings['overall_savings_percentage']:.2f}%")
print(f"Average ROI Period:             {total_savings['average_roi_months']:.1f} months")
print(f"CO2 Reduction (Annual):         {total_savings['co2_reduction_kg_annual']:,.2f} kg")
print('='*60)

In [ ]:
# Priority recommendations (quick wins)
priority = optimization.get_priority_recommendations(recommendations, n=5, max_roi_months=24)

print('\n--- Priority Actions (ROI <= 24 months) ---\n')
display(priority[['Device', 'Category', 'Recommendation',
                  'Monthly_Savings_Rp', 'Estimated_Cost_Rp', 'ROI_Months']])

In [ ]:
# Scenario analysis
scenarios = optimization.calculate_all_scenarios(df_devices)

print('\n--- Optimization Scenarios Analysis ---\n')
display(scenarios[['scenario_description', 'applicable_devices',
                   'current_monthly_kwh', 'potential_savings_kwh',
                   'savings_percentage', 'annual_savings_rp']])

## 7. Export Results

### Save All Tables, Figures, and Model

In [ ]:
# Export processed data
print('Exporting results...\n')

# 1. Processed device data
processed_path = data_input.export_processed_data(df_devices)
print(f'Processed data saved: {processed_path}')

# 2. Summary tables
summary_path = energy_calculator.export_summary_tables(df_devices)
print(f'Summary tables saved: {summary_path}')

# 3. Optimization recommendations
rec_path = optimization.export_recommendations(recommendations, total_savings, scenarios)
print(f'Recommendations saved: {rec_path}')

# 4. Save model
model_path = xgboost_model.save_model(model)
print(f'Trained model saved: {model_path}')

In [ ]:
# List all exported files
print('\n--- Exported Files Summary ---\n')

# Figures
print('Figures (outputs/figures/):')
for f in config.FIGURES_DIR.glob('*.png'):
    print(f'   - {f.name}')

# Tables
print('\nTables (outputs/tables/):')
for f in config.TABLES_DIR.glob('*.xlsx'):
    print(f'   - {f.name}')

# Processed data
print('\nProcessed Data (data/processed/):')
for f in config.PROCESSED_DATA_DIR.glob('*.xlsx'):
    print(f'   - {f.name}')

## 8. Conclusion

### Summary of Key Findings

In [ ]:
print('\n' + '='*70)
print('EXECUTIVE SUMMARY - ENERGY ANALYSIS BTI CAMPUS BUILDING')
print('='*70)

print('\nCURRENT ENERGY CONSUMPTION:')
print(f'   Total Daily:   {total_daily_kwh:,.2f} kWh (Rp {daily_cost:,.0f})')
print(f'   Total Monthly: {total_monthly_kwh:,.2f} kWh (Rp {monthly_cost:,.0f})')
print(f'   Total Annual:  {total_annual_kwh:,.2f} kWh (Rp {annual_cost:,.0f})')

print('\nTOP ENERGY CONSUMERS:')
for i, row in top_consumers.head(3).iterrows():
    print(f"   {i}. {row['Device_Name']}: {row['Monthly_Energy_kWh']:,.2f} kWh/month")

print('\nXGBOOST MODEL PERFORMANCE:')
print(f"   R2 Score: {metrics['r2']:.4f}")
print(f"   RMSE: {metrics['rmse']:.4f} kWh")
print(f"   MAPE: {metrics['mape']:.2f}%")

print('\nOPTIMIZATION POTENTIAL:')
print(f"   Potential Annual Savings: {total_savings['potential_annual_savings_kwh']:,.2f} kWh")
print(f"   Potential Cost Savings: Rp {total_savings['potential_annual_savings_rp']:,.0f}")
print(f"   Overall Reduction: {total_savings['overall_savings_percentage']:.2f}%")
print(f"   Average ROI: {total_savings['average_roi_months']:.1f} months")

print('\nTOP RECOMMENDATIONS:')
for i, row in priority.head(3).iterrows():
    print(f"   {i}. {row['Device']}: {row['Recommendation'][:50]}...")

print('\n' + '='*70)
print('Analysis completed successfully!')
print('All results have been exported for journal publication.')
print('='*70)

### Next Steps

1. **Data Verification**: Replace sample data with actual BTI building device inventory
2. **Model Refinement**: Collect real consumption data for model validation
3. **Implementation**: Prioritize recommendations based on ROI and feasibility
4. **Monitoring**: Set up energy monitoring system to track improvements

### How to Use Results in Journal

- All figures in `outputs/figures/` are 300 DPI PNG files ready for publication
- Tables in `outputs/tables/` can be imported into Word/LaTeX
- Metrics and statistics can be cited directly from this notebook
- The trained model can be used for real-time energy prediction

---

**End of Analysis**